# Records: build once, reopen by key

A **record** is a directory that holds the result of one computation. Its name is the SHA-256
**key** of three things: the record type, a typed **spec** (the parameters) and the **inputs**
passed to the build. Asking for a record reopens it if it exists and builds it otherwise.

This notebook defines a record type, builds and reopens a record, reads what it stores, and
shows which changes give a new key.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
from jaxtyping import Float, Int

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-records-"))
td.set_root(root)  # every record in this notebook goes under this temporary directory

## Specs

A `Spec` subclass becomes a frozen, keyword-only dataclass. Its fields are the parameters that
identify a result. Two field options control the key:

- `td.field(default, hashed=False)` keeps a field out of the key, for settings that do not change
  the result (device, number of workers).
- `td.field(default, since=v)` keeps a field out of the key while it equals its default, so
  adding the field later leaves existing keys unchanged.

In [ ]:
class Optimizer(td.Spec):
    lr: float = 0.1
    n_iterations: int = 20


class KMeansSpec(td.Spec):
    n_clusters: int
    seed: int = 0
    optimizer: Optimizer = td.field(default_factory=Optimizer)  # nested spec
    device: str = td.field("cpu", hashed=False)


spec = KMeansSpec(n_clusters=3)
spec  # the notebook repr lists every field and marks fields outside the key

Specs are values: `replace` returns a modified copy (nested names use `__` or dots), `flat`
lists every field by dotted name, and `diff` compares two specs.

In [ ]:
other = spec.replace(seed=1, optimizer__lr=0.01, device="cuda")
print(other.flat())
print(spec.diff(other))
print("key-relevant only:", spec.diff(other, hashed=True))

Canonical JSON makes keys stable: a float field given an int is stored as a float, and the
unhashed `device` does not enter the key.

In [ ]:
assert KMeansSpec(n_clusters=3, optimizer=Optimizer(lr=1)).key() == KMeansSpec(
    n_clusters=3, optimizer=Optimizer(lr=1.0), device="cuda"
).key()
spec.hashed_dict()

## A record type

A `Record` subclass names its type with `__record_name__` (the key uses it, so set it; the
fallback `module:qualname` changes when a class moves and is `__main__:...` in a notebook),
declares its `Spec`, annotates the fields it stores, and implements `build`.

jaxtyping annotations name the axes of array fields: a size `n_<things>` becomes the dimension
`<thing>`, so `"n_points n_dims"` is stored as `("point", "dim")`.

In [ ]:
builds = []  # counts how often `build` runs in this process


class Points(td.Record):
    __record_name__ = "Points"

    class Spec(td.Spec):
        n_points: int = 300
        seed: int = 0

    xy: Float[np.ndarray, "n_points n_dims"]

    def build(self, spec):
        rng = np.random.default_rng(spec.seed)
        centres = np.array([[-2.0, 0.0], [2.0, 0.0], [0.0, 3.0]])
        self.xy = centres[rng.integers(0, 3, spec.n_points)] + rng.normal(scale=0.6, size=(spec.n_points, 2))


class KMeans(td.Record):
    __record_name__ = "KMeans"
    Spec = KMeansSpec

    centres: Float[np.ndarray, "n_clusters n_dims"]
    labels: Int[np.ndarray, "n_points"]

    def build(self, spec, points: Points):
        builds.append(spec)
        xy = points.xy
        rng = np.random.default_rng(spec.seed)
        centres = xy[rng.choice(len(xy), spec.n_clusters, replace=False)]
        for _ in range(spec.optimizer.n_iterations):
            labels = np.argmin(((xy[:, None] - centres[None]) ** 2).sum(-1), axis=1)
            target = np.stack([xy[labels == k].mean(0) for k in range(spec.n_clusters)])
            centres = centres + spec.optimizer.lr * 10 * (target - centres)
        self.centres = centres
        self.labels = labels
        self["summary"] = {"inertia": float(((xy - centres[labels]) ** 2).sum())}  # stored as JSON

## Build and reopen

`Record.get(spec, **inputs)` returns a finished record. The spec can be a `Spec` instance, a
mapping, or nothing (all defaults). Keyword arguments beyond the spec are **inputs**: they reach
`build` and enter the key. A record input enters by its type and key, an array by a digest of
its bytes, other values by their canonical JSON form.

In [ ]:
points = Points.get()
fit = KMeans.get(KMeans.Spec(n_clusters=3), points=points)  # builds
again = KMeans.get({"n_clusters": 3}, points=Points.get())   # reopens: same type, spec and inputs
print(len(builds), "build;", "same record:", again == fit)
print(fit.status, fit.key[:16], fit.path.relative_to(root))

The notebook repr shows the spec and the stored fields; `tree()` prints the same as text.

In [ ]:
fit

In [ ]:
fit.tree()

## Read fields

- `record.name` returns the declared type: numpy for jaxtyping array fields.
- `record["name"]` returns a lazy handle: a `zarr.Array` for arrays, nothing is read yet.
- `record.load("name")` always reads into memory.
- `record.to_xarray("name")` labels an array with its stored dimension names.

In [ ]:
print(type(fit.centres).__name__, fit.centres.shape)
print(type(fit["centres"]).__name__, fit["centres"].shape)
print(fit.summary, fit.keys(), "labels" in fit)
fit.to_xarray("centres")

`record.meta` is the content of `_meta.json`: the full key, the stored spec (including
unhashed fields), the inputs by identity, status, timestamps, the field manifest and provenance
(host, user, job id, package versions, git commit of the repository defining the class).

In [ ]:
print(sorted(fit.meta))
print(fit.meta["inputs"])
print(fit.meta["spec"]["device"], fit.spec.optimizer.lr)
print(sorted(fit.meta["provenance"]))

## Which changes give a new key

The key covers the record type, `schema_version`, the hashed spec fields and the inputs. The
unhashed `device` field does not change it; another `Points` input does.

In [ ]:
gpu = KMeans.get({"n_clusters": 3, "device": "cuda"}, points=points)
print("device changes the key:", gpu.key != fit.key, "| builds so far:", len(builds))

other_points = Points.get({"seed": 1})
refit = KMeans.get({"n_clusters": 3}, points=other_points)
print("another input changes the key:", refit.key != fit.key, "| builds so far:", len(builds))

Arrays passed as inputs are hashed by digest. Use `td.Digest.of(array)` to put the same
fingerprint in a spec field.

In [ ]:
weights = np.linspace(0.5, 1.5, 300)
k1 = KMeans.compute_key({"n_clusters": 3}, points=points, weights=weights)  # no filesystem access
k2 = KMeans.compute_key({"n_clusters": 3}, points=points, weights=weights.copy())
k3 = KMeans.compute_key({"n_clusters": 3}, points=points, weights=weights + 1e-9)
print(k1 == k2, k1 == k3)
td.Digest.of(weights)

## Look without building

- `locate(spec, **inputs)` returns the record handle whatever its status; nothing is built.
- `lookup(spec, **inputs)` returns the finished record or `None`.
- `compute_key(spec, **inputs)` returns the full key without touching the filesystem.

In [ ]:
missing = KMeans.locate({"n_clusters": 5}, points=points)
print(missing.status, KMeans.lookup({"n_clusters": 5}, points=points))
print(KMeans.lookup({"n_clusters": 3}, points=points) == fit)

## Finished records are read-only

A key promises its contents, so a finished record refuses writes. Change the spec instead, or
use a workspace record (`mutable = True`, see the notebook on classes that own a record).

In [ ]:
try:
    fit.centres = np.zeros((3, 2))
except td.ReadOnlyError as exc:
    print("ReadOnlyError:", exc)

## Adding a field without orphaning records

Suppose `KMeans` later gains a `tolerance` parameter. Declared with `since=`, the field stays
out of the key while it equals its default, so existing records keep their keys. Queries and
tables show the default for records built before the field existed.

In [ ]:
class KMeansSpec2(KMeansSpec):
    tolerance: float = td.field(0.0, since=2)


class KMeans(KMeans):  # the same record type, redefined with the new spec
    __record_name__ = "KMeans"
    Spec = KMeansSpec2


print(KMeans.compute_key({"n_clusters": 3}, points=points) == fit.key)
print(KMeans.compute_key({"n_clusters": 3, "tolerance": 1e-4}, points=points) == fit.key)
print(KMeans.lookup({"n_clusters": 3}, points=points).spec)

A change to `build` that invalidates old results calls for a new `schema_version`: it enters
every key of the type, so old records are no longer found.

In [ ]:
class KMeansV1(KMeans):
    __record_name__ = "KMeans"
    schema_version = 1


print(KMeansV1.lookup({"n_clusters": 3}, points=points))

## Hand-named directories, deletion

`Record.at(path, spec)` builds or opens a record at a path you choose, as datamate's
`Directory(path, config)` did. `td.Record.at(path)` opens any record and returns it as its
registered class.

In [ ]:
named = Points.at(root / "runs" / "reference", {"n_points": 50})
reopened = td.Record.at(root / "runs" / "reference")
print(type(reopened).__name__, reopened.spec, reopened.xy.shape)

`record.delete()` removes the directory; the next `get` builds it again.

In [ ]:
fit.delete()
print(fit.status)
fit = KMeans.get({"n_clusters": 3}, points=points)
print(fit.status, "| builds so far:", len(builds))

## On disk

Records live under `<root>/<record name>/<first 16 hex digits of the key>/`; `.tensordir/`
holds leases, temporary builds and the index.

In [ ]:
for path in sorted(root.rglob("*")):
    depth = len(path.relative_to(root).parts)
    if depth <= 3 and "data.zarr" not in path.parts[:-1] and not path.name.startswith("zarr"):
        print("    " * (depth - 1) + path.name + ("/" if path.is_dir() else ""))